# Using featsel, end to end

What a user does with the package, from one selector to a full comparison grid.

It runs on synthetic data, so it takes seconds and the truly informative
features are known. SCAN-B works the same way, loaded through a config file
(last section).

**Built today:** selector classes, selectors inside a scikit-learn `Pipeline`,
the in-memory runner with parallel splits and optional ground truth, and the
analysis helpers.

**Planned, not built:** selectors as full scikit-learn estimators (so
`cross_val_score` and `GridSearchCV` can clone them), Higher Criticism, and
nested tuning of the model's regularization.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from featsel import kuncheva_index, summarize
from featsel.run import run
from featsel.selectors import ANOVAFSelector

# 300 samples, 2000 features, the first 20 informative (shuffle=False keeps them first).
X, y = make_classification(n_samples=300, n_features=2000, n_informative=20,
                           n_redundant=0, shuffle=False, random_state=0)
truth = np.arange(20)

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7,
                                                    stratify=y, random_state=0)
X.shape

(300, 2000)

## 1. One selector

Import the selector class and make an instance. A misspelled class name fails
at import, and an editor completes it.

In [2]:
selector = ANOVAFSelector(n_features=20)
selector.fit(X_train, y_train)

chosen = selector.get_support(indices=True)
print('chosen features   :', chosen)
print('truly informative :', np.intersect1d(chosen, truth).size, 'of', chosen.size)
print('reduced shape     :', selector.transform(X_test).shape)

chosen features   : [1117   14  585    1  576 1578 1331  268  439  134    4  478 1785 1344
  997    8  831 1161   17    2]
truly informative : 6 of 20
reduced shape     : (90, 20)


## 2. Inside a scikit-learn pipeline

Selection is fitted on the training part only, as a pipeline step.

In [3]:
pipe = make_pipeline(StandardScaler(),
                     ANOVAFSelector(n_features=20),
                     LogisticRegression(max_iter=1000))
pipe.fit(X_train, y_train)
print(f'test accuracy: {pipe.score(X_test, y_test):.3f}')

test accuracy: 0.667


## 3. A full comparison, in memory

The protocol is a plain dict. `run` takes `X, y` from memory, here with the
ground truth, runs the splits on two worker processes and writes one row per
(split, selector, k, model) to a CSV.

Here selectors are named as text, because the same dict can come from a YAML
file. The runner turns each name into its class.

The runner still names selectors by the old wrapper's method names
(`random`, `anova_f`). Switching it to class names is part of the planned
conversion.

In [4]:
config = {
    'selectors': [{'name': 'random', 'k': [10, 20, 50]},
                  {'name': 'anova_f', 'k': [10, 20, 50]}],
    'models': [{'name': 'logistic_regression', 'params': {'max_iter': 1000}},
               {'name': 'knn'}],
    'n_splits': 10,
    'train_sizes': [0.7],
    'n_jobs': 2,
    'output': '../results/demo_usage.csv',
}

results = pd.read_csv(run(X, y, config, truth=truth))
results.shape


=== ../results/demo_usage.csv ===
Dataset      : 300 samples x 2000 features
Seed         : 42
Tasks        : 1
  multiclass             2 classes (0 150, 1 150)
Splits       : 10 stratified Monte Carlo draws
  train_size 0.7: 210 train / 90 validation
Preprocess   : standard
Metrics      : accuracy, macro_f1, balanced_accuracy, mcc, g_mean
Workers      : 2 (parallel over splits)
Selectors    :
  random           k = 10 (0.50%), 20 (1.00%), 50 (2.50%)
  anova_f          k = 10 (0.50%), 20 (1.00%), 50 (2.50%)
Models       :
  logistic_regression {'max_iter': 1000}
  knn              {}

Total        : 120 executions (1 tasks x 10 splits x 1 train sizes x 1 preprocess x 6 selector-k x 2 models)



  0%|          | 0/120 [00:00<?, ?exec/s]

Wrote 120 executions to ../results/demo_usage.csv in 4s


(120, 25)

## 4. Reading the results

`summarize` gives the median and interquartile range over splits for every
cell. `kuncheva_index` measures how stable the chosen features are across
splits: 1 is identical every time, 0 is chance.

In [5]:
summary = summarize(results)
summary[['selector', 'k', 'model', 'macro_f1_median', 'macro_f1_q1', 'macro_f1_q3',
         'selection_time_s_median']].round(3)

,selector,k,model,macro_f1_median,macro_f1_q1,macro_f1_q3,selection_time_s_median
0,anova_f,10,knn,0.644,0.614,0.662,0.025
1,anova_f,10,logistic_regression,0.654,0.621,0.686,0.025
2,anova_f,20,knn,0.578,0.568,0.624,0.025
3,anova_f,20,logistic_regression,0.644,0.603,0.688,0.025
4,anova_f,50,knn,0.599,0.563,0.638,0.022
5,anova_f,50,logistic_regression,0.614,0.567,0.645,0.022
6,random,10,knn,0.533,0.522,0.556,0.008
7,random,10,logistic_regression,0.477,0.466,0.489,0.008
8,random,20,knn,0.507,0.472,0.538,0.011
9,random,20,logistic_regression,0.475,0.467,0.525,0.011


In [6]:
kuncheva_index(results, n_total_features=X.shape[1])[
    ['selector', 'k', 'consistency_index', 'n_pairs']].round(3)

,selector,k,consistency_index,n_pairs
0,anova_f,10,0.453,45
1,anova_f,20,0.402,45
2,anova_f,50,0.349,45
3,random,10,0.002,45
4,random,20,0.006,45
5,random,50,0.003,45


Because the truth is known, every row also records how many truly
informative features were found (recall) and how much of the selection is
truly informative (precision).

In [7]:
(results.drop_duplicates(['split', 'selector', 'k'])
        .groupby(['selector', 'k'])[['truth_recall', 'truth_precision']]
        .median().round(3))

truth_recall  truth_precision
selector k                                
anova_f  10         0.250            0.500
         20         0.325            0.325
         50         0.400            0.160
random   10         0.000            0.000
         20         0.000            0.000
         50         0.000            0.000

## 5. The same from the command line

For data on disk, a YAML file holds the same dict plus a `dataset` entry that
names a dataset config. Paths are relative to the config file.

```bash
python -m featsel.run --config configs/experiment_classifiers.yaml --n-jobs 12
python -m featsel.run --config configs/experiment_classifiers.yaml --resume
```

`--resume` skips every row already in the output CSV, so a killed run
continues where it stopped.